# AI-Powered SQL Database Query Application


In [2]:
import sqlite3
import re
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.utilities import SQLDatabase
from langchain_chroma import  Chroma


print("Libraries loaded successfully!")

/var/folders/jt/35j9k5_x051f8pzqkqp505xh0000gn/T/ipykernel_99906/678994698.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Libraries loaded successfully!


In [4]:
conn = sqlite3.connect("itm_university.db")
cursor = conn.cursor()


cursor.executescript("""
DROP TABLE IF EXISTS students;

CREATE TABLE students (
    student_id TEXT PRIMARY KEY,
    name TEXT,
    branch TEXT,
    semester INTEGER,
    cgpa REAL,
    attendance_pct INTEGER,
    fees_status TEXT
);

INSERT INTO students VALUES 
('ITM101', 'Aarav Sharma', 'CSE', 7, 8.9, 82, 'Paid'),
('ITM102', 'Priya Patel', 'CSE', 7, 9.4, 91, 'Paid'),
('ITM103', 'Rohan Verma', 'Cyber Security', 5, 7.8, 68, 'Pending'),
('ITM104', 'Ananya Iyer', 'AI & DS', 7, 9.6, 95, 'Paid'),
('ITM105', 'Vikram Singh', 'CSE', 5, 8.2, 71, 'Paid'),
('ITM106', 'Sneha Kulkarni', 'Cyber Security', 3, 8.7, 88, 'Paid'),
('ITM107', 'Rahul Nair', 'CSE', 7, 9.1, 74, 'Paid');
""")
conn.commit()
conn.close()

db = SQLDatabase.from_uri("sqlite:///itm_university.db")
print("Updated Schema:\n", db.get_table_info())

Updated Schema:
 
CREATE TABLE students (
	student_id TEXT, 
	name TEXT, 
	branch TEXT, 
	semester INTEGER, 
	cgpa REAL, 
	attendance_pct INTEGER, 
	fees_status TEXT, 
	PRIMARY KEY (student_id)
)

/*
3 rows from students table:
student_id	name	branch	semester	cgpa	attendance_pct	fees_status
ITM101	Aarav Sharma	CSE	7	8.9	82	Paid
ITM102	Priya Patel	CSE	7	9.4	91	Paid
ITM103	Rohan Verma	Cyber Security	5	7.8	68	Pending
*/


In [5]:
sql_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an expert SQL assistant for ITM Skills University college database.
Convert the user's question into ONE valid SQLite query.

DATABASE SCHEMA:
{table_info}

COLUMN NOTES:
- cgpa: cumulative grade point, scale of 10
- attendance_pct: attendance percentage, scale of 0-100
- semester: integer 1-8 (e.g. 7th semester = 4th year)
- branch: engineering branch (e.g. 'CSE', 'AI & DS', 'Cyber Security')

RULES:
1. Generate ONLY a SELECT query. Never use INSERT, UPDATE, DELETE, DROP, ALTER.
2. Use ONLY tables and columns present in the schema above.
3. Limit results to 20 rows unless the user explicitly asks for more.
4. If the question is ambiguous or cannot be answered from the schema, return exactly: CANNOT_ANSWER
5. Output ONLY the raw SQL query. No markdown formatting (```sql), no explanation, no extra text.
6. When the user asks a follow-up question, use the conversation history to understand context.

FEW-SHOT EXAMPLES:
Q: CSE ke top 5 students by CGPA
SQL: SELECT name, cgpa FROM students WHERE branch = 'CSE' ORDER BY cgpa DESC LIMIT 5;

Q: Kitne students ki attendance 75% se kam hai?
SQL: SELECT COUNT(*) FROM students WHERE attendance_pct < 75;

Q: Unke naam kya hain?
SQL: SELECT name, attendance_pct FROM students WHERE attendance_pct < 75;
"""
    ),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

print("SQL prompt configured!")

SQL prompt configured!


In [6]:
llm = ChatOllama(model="llama3.2", temperature=0.0)

chat_histories = {}
def get_session_history(session_id: str):
    if session_id not in chat_histories:
        chat_histories[session_id] = InMemoryChatMessageHistory()
    return chat_histories[session_id]

# Combine prompt + LLM
sql_chain = sql_prompt | llm

# Wrap with memory
sql_generator = RunnableWithMessageHistory(
    sql_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

print("SQL Generator with Memory is ready!")

SQL Generator with Memory is ready!


/Users/ashutoshrai/.local/share/uv/python/cpython-3.11.14-macos-aarch64-none/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
def clean_sql(raw_output: str) -> str:
    # Strip markdown if model adds ```sql ... ```
    cleaned = re.sub(r"```(sql)?", "", raw_output).strip()
    return cleaned

def ask_college_assistant(session_id: str, question: str) -> str:
    table_info = db.get_table_info()
    
    # 1. Generate SQL with memory
    response = sql_generator.invoke(
        {
            "table_info": table_info,
            "input": question
        },
        config={"configurable": {"session_id": session_id}}
    )
    
    query = clean_sql(response.content)
    print(f"\n[Generated SQL]: {query}")
    
    if query == "CANNOT_ANSWER":
        return "Sorry, this question cannot be answered from the university database."
    
    # 2. Execute SQL
    try:
        db_result = db.run(query)
        print(f"[Query Result]: {db_result}")
    except Exception as e:
        return f"Database error: {e}"
    
    # 3. Synthesize natural answer
    synthesis_prompt = f"""
    User Question: "{question}"
    Database Result: {db_result}

    Provide a concise, helpful answer in the same language/tone (English/Hinglish) as the user asked.
    """
    answer = llm.invoke(synthesis_prompt)
    return answer.content

In [8]:
session = "student_session_01"

# Question 1 (Hinglish):
print(ask_college_assistant(session, "CSE ke top 3 students by CGPA kaun hain?"))

# Question 2 (Follow-up using memory: "Inki attendance kitni hai?"):
print(ask_college_assistant(session, "Inki attendance percentage kitni hai?"))

# Question 3:
print(ask_college_assistant(session, "Kitne students ki attendance 75% se kam hai?"))

# Question 4 (Follow-up: "Unke naam kya hain?"):
print(ask_college_assistant(session, "Unke naam aur branch batao"))

/var/folders/jt/35j9k5_x051f8pzqkqp505xh0000gn/T/ipykernel_99906/2724508707.py:6: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  chat_histories[session_id] = InMemoryChatMessageHistory()



[Generated SQL]: SELECT name, cgpa FROM students WHERE branch = 'CSE' ORDER BY cgpa DESC LIMIT 3;
[Query Result]: [('Priya Patel', 9.4), ('Rahul Nair', 9.1), ('Aarav Sharma', 8.9)]
CSE ke top 3 students by CGPA hain:

1. Priya Patel - 9.4
2. Rahul Nair - 9.1
3. Aarav Sharma - 8.9

[Generated SQL]: SELECT attendance_pct FROM students WHERE branch = 'CSE' ORDER BY attendance_pct DESC LIMIT 3;
[Query Result]: [(91,), (82,), (74,)]
"Attendance percentage of Inki is 91%."

[Generated SQL]: SELECT COUNT(*) FROM students WHERE attendance_pct < 75;
[Query Result]: [(3,)]
"Kisi bhi student ki attendance 75% se kam hai, ek hi student hai jiska attendance 3 students kee attendance se kam hai."

[Generated SQL]: SELECT name, branch FROM students WHERE attendance_pct < 75;
[Query Result]: [('Rohan Verma', 'Cyber Security'), ('Vikram Singh', 'CSE'), ('Rahul Nair', 'CSE')]
Unke naam aur branch yeh hain:

1. Rohan Verma - Cyber Security
2. Vikram Singh - CSE
3. Rahul Nair - CSE
